# 02 — Subcluster Cards (Per-Tissue Report)

Generate an **interactive HTML report** and a **PowerPoint deck** (one slide per subcluster)
from the outputs of `01_disease_subcluster_detection.ipynb`.

Run this notebook **once per tissue**, using the same `RESULTS_DIR` / `TISSUE` you used in Notebook 01.

**Workflow:**
1. Load the AnnData object and DE results from Notebook 01
2. Generate UMAP panels, volcano plots, and functional enrichment (g:Profiler)
3. Render interactive HTML report with sidebar navigation and paginated DE tables
4. Render PowerPoint deck (one slide per subcluster)

**Outputs (all written to `{OUTPUT_DIR}/{TISSUE}_subcluster_report/`):**
- `index.html` — interactive offline HTML report (keep `img/` and `de/` alongside it)
- `img/{safe}/` — per-subcluster PNG files referenced by the HTML
- `de/` — per-subcluster DE CSV files (downloadable from the report)
- `{OUTPUT_DIR}/{TISSUE}_subcluster_cards.pptx` — static PowerPoint deck

## 1. Imports

In [ ]:
import base64
import io
import shutil
from datetime import datetime
from pathlib import Path

import anndata as ad
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
from anndata import AnnData
from jinja2 import Environment, FileSystemLoader
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Cm, Inches, Pt

import scutils
from scutils.plotting.dotplots import create_pathway_dotplot
from scutils.plotting.volcano_plot import volcano_plot
from scutils.preprocessing.pseudobulk import format_deseq2_results
from scutils.tools.functional import get_enriched_terms

matplotlib.use("Agg")

## 2. Configuration

Key parameters:
- `ADATA_PATH` — `.h5ad` file used in Notebook 01 (required for UMAP panels)
- `RESULTS_DIR` — output directory from Notebook 01 for this tissue (contains `de/`, `enrichment/`)
- `TISSUE` — tissue/dataset label (used in filenames and report headers)
- `SUBCLUSTER_KEY` — `adata.obs` column that holds subcluster labels (default: `"disease_subcluster"`)
- `DISEASE_KEY` / `CELLTYPE_KEY` — `adata.obs` columns
- `ENRICHMENT_ORGANISM` — g:Profiler organism code (default `"hsapiens"`)
- `FORCE_RERUN_ENRICHMENT` — re-run g:Profiler even if cached CSV already exists
- `OUTPUT_DIR` — where HTML and PPTX are saved (defaults to `RESULTS_DIR`)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input ─────────────────────────────────────────────────────────────────────
ADATA_PATH    : Path = Path("data/my_tissue.h5ad")
RESULTS_DIR   : Path = Path("results/disease_subclusters/my_tissue")
TISSUE        : str  = "my_tissue"

# ── AnnData keys ──────────────────────────────────────────────────────────────
SUBCLUSTER_KEY : str = "disease_subcluster"
CELLTYPE_KEY   : str = "cell_type"
DISEASE_KEY    : str = "disease"

# ── DE columns expected in the CSV (must match Notebook 01 output) ─────────────
DE_PVAL_COL     : str = "padj"
DE_LFC_COL      : str = "log2FoldChange"
GENE_SYMBOL_COL : str = "gene"

# ── Volcano plot settings ─────────────────────────────────────────────────────
VOLCANO_PVAL_CUTOFF : float = 0.05
VOLCANO_LFC_CUTOFF  : float = 0.5
VOLCANO_TOP_N_UP    : int   = 10
VOLCANO_TOP_N_DOWN  : int   = 5
VOLCANO_FIGSIZE     : tuple = (10, 6)

# ── Functional enrichment settings ───────────────────────────────────────────
ENRICHMENT_ORGANISM     : str         = "hsapiens"        # g:Profiler organism code
FORCE_RERUN_ENRICHMENT  : bool        = False             # re-run even if CSV exists
ENRICHMENT_MAX_PATHWAYS : int         = 15
ENRICHMENT_PVAL         : float       = 0.05
ENRICHMENT_SOURCES      : list | None = None              # None = all sources; e.g. ["KEGG","WP"]
ENRICHMENT_FIGSIZE      : tuple       = (10, 6)
PATHWAY_COLORS          : dict        = {
    "GO:BP": "#1f77b4",
    "GO:MF": "#ff7f0e",
    "GO:CC": "#2ca02c",
    "KEGG":  "#d62728",
    "REAC":  "#9467bd",
    "WP":    "#CD853F",
}

# ── UMAP settings ─────────────────────────────────────────────────────────────
UMAP_FIGSIZE        : tuple = (6, 5)
UMAP_POINT_SIZE     : float = None   # None = auto
BACKGROUND_COLOR    : str   = "#d3d3d3"

# ── Output ────────────────────────────────────────────────────────────────────
# HTML report is written to a subdirectory; images are stored per-subcluster:
#   {OUTPUT_DIR}/{TISSUE}_subcluster_report/
#       index.html                    ← open this in a browser
#       img/{safe}/umap_celltype.png  ← per-subcluster image directories
#       img/{safe}/umap_disease.png
#       img/{safe}/umap_highlight.png
#       img/{safe}/volcano.png
#       img/{safe}/enrichment.png
#       de/{safe}.csv
OUTPUT_DIR : Path = RESULTS_DIR

# ── PPTX layout ───────────────────────────────────────────────────────────────
PPTX_TOP_DE_GENES : int = 20

# ─────────────────────────────────────────────────────────────────────────────
REPORT_DIR     : Path = OUTPUT_DIR / f"{TISSUE}_subcluster_report"
REPORT_IMG_DIR : Path = REPORT_DIR / "img"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_IMG_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")
print(f"  Tissue      : {TISSUE}")
print(f"  Results dir : {RESULTS_DIR}")
print(f"  Report dir  : {REPORT_DIR}")

## 3. Load Data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print(f"Loaded: {adata}")

# Validate required keys
for key, attr in [
    (SUBCLUSTER_KEY, "obs"),
    (CELLTYPE_KEY,   "obs"),
    (DISEASE_KEY,    "obs"),
]:
    assert key in getattr(adata, attr).columns, (
        f"Key '{key}' not found in adata.{attr}. "
        f"Available: {list(getattr(adata, attr).columns)}"
    )

info_key = f"{SUBCLUSTER_KEY}_info"
assert info_key in adata.uns, (
    f"'{info_key}' not found in adata.uns. "
    "Ensure you ran 01_disease_subcluster_detection.ipynb first."
)

subcluster_info: pd.DataFrame = adata.uns[info_key].copy()
background_mask = adata.obs[SUBCLUSTER_KEY] == "background"

# All non-background subcluster labels
subcluster_labels = [
    lbl for lbl in adata.obs[SUBCLUSTER_KEY].unique()
    if lbl != "background"
]
print(f"Found {len(subcluster_labels)} disease-enriched subclusters:")
for lbl in sorted(subcluster_labels):
    print(f"  {lbl}")


## 4. Helper Functions

In [ ]:
import re as _re

# ---------------------------------------------------------------------------
# Safe-label helpers
# ---------------------------------------------------------------------------

_SAFE_LABEL_CACHE: dict[str, str] = {}


def _abbrev_words(text: str, chars_per_word: int = 3) -> str:
    """Return first ``chars_per_word`` chars of each whitespace-separated word."""
    return "_".join(w[:chars_per_word] for w in text.split() if w)


def _safe_label_raw(label: str, max_len: int = 35) -> str:
    """Abbreviate the cell-type part, keep disease + sub-ID as intact as possible.

    Labels are expected in the form ``"{cell_type}|{disease}|sub{N}"``.
    The cell type is abbreviated to the first 3 characters of each word.
    The sub-ID (last segment) is always preserved in full; the disease segment
    is truncated only if the total would exceed ``max_len``.
    """
    if "|" in label:
        parts = [p.strip() for p in label.split("|")]
        ct_abbrev = _re.sub(r"[^\w\-]", "_", _abbrev_words(parts[0]))
        ct_abbrev = _re.sub(r"_+", "_", ct_abbrev).strip("_")

        rest_parts = [_re.sub(r"[^\w\-]", "_", p) for p in parts[1:]]
        suffix = "_".join(rest_parts)
        safe = f"{ct_abbrev}_{suffix}"

        if len(safe) > max_len and len(rest_parts) >= 2:
            # Always keep the last segment (sub-ID) intact; truncate disease
            sub_id = rest_parts[-1]
            disease = "_".join(rest_parts[:-1])
            budget = max_len - len(ct_abbrev) - 2 - len(sub_id)  # 2 underscores
            disease = disease[:max(0, budget)].rstrip("_")
            safe = f"{ct_abbrev}_{disease}_{sub_id}"
    else:
        safe = _re.sub(r"[^\w\-]", "_", label)

    safe = _re.sub(r"_+", "_", safe).strip("_")
    return safe[:max_len].rstrip("_")


def _safe_label(label: str) -> str:
    """Return a filesystem-safe, URL-safe, \u226435-char label.

    Uses the pre-computed collision-free map populated by
    ``_build_safe_label_map``; falls back to ``_safe_label_raw`` if the map
    has not been built yet (e.g. inside ``_load_de`` before the asset loop).
    """
    if label in _SAFE_LABEL_CACHE:
        return _SAFE_LABEL_CACHE[label]
    return _safe_label_raw(label)


def _build_safe_label_map(labels: list[str]) -> None:
    """Pre-compute a collision-free ``{original_label: safe_label}`` mapping.

    Call this **once** after all subcluster labels are known, before the asset
    generation loop.  Populates ``_SAFE_LABEL_CACHE`` in-place; subsequent
    calls to ``_safe_label`` will use the cached results.

    If two labels produce the same abbreviated form, the second receives a
    numeric suffix (``_2``, ``_3``, \u2026).
    """
    _SAFE_LABEL_CACHE.clear()
    used: set[str] = set()
    for label in labels:
        base = _safe_label_raw(label)
        if base not in used:
            _SAFE_LABEL_CACHE[label] = base
            used.add(base)
        else:
            i = 2
            stem = base[:32].rstrip("_")
            candidate = f"{stem}_{i}"
            while candidate in used:
                i += 1
                candidate = f"{stem}_{i}"
            _SAFE_LABEL_CACHE[label] = candidate
            used.add(candidate)
    print(f"Safe-label map built: {len(labels)} subclusters.")

def _make_umap_celltype(adata: AnnData, figsize: tuple) -> plt.Figure:
    """UMAP coloured by cell type."""
    fig, ax = plt.subplots(figsize=figsize)
    sc.pl.umap(adata, color=CELLTYPE_KEY, ax=ax, show=False,
               frameon=False, title="Cell type")
    return fig


def _make_umap_disease(adata: AnnData, figsize: tuple) -> plt.Figure:
    """UMAP coloured by disease condition."""
    fig, ax = plt.subplots(figsize=figsize)
    sc.pl.umap(adata, color=DISEASE_KEY, ax=ax, show=False,
               frameon=False, title="Disease")
    return fig


def _make_umap_highlight(
    adata: AnnData,
    subcluster_label: str,
    figsize: tuple,
) -> plt.Figure:
    """UMAP with subcluster cells highlighted; background cells in grey."""
    mask = adata.obs[SUBCLUSTER_KEY] == subcluster_label

    fig, ax = plt.subplots(figsize=figsize)
    coords = adata.obsm["X_umap"]
    pt_size = UMAP_POINT_SIZE or max(120000 / adata.n_obs, 0.5)
    ax.scatter(
        coords[~mask, 0], coords[~mask, 1],
        s=pt_size, c=BACKGROUND_COLOR, linewidths=0, rasterized=True,
    )
    ax.scatter(
        coords[mask, 0], coords[mask, 1],
        s=pt_size * 1.5, c="#e63946", linewidths=0, rasterized=True, zorder=2,
    )
    ax.set_title(f"Subcluster: {subcluster_label}", fontsize=10)
    ax.axis("off")
    return fig


def _fmt_float(v, decimals: int = 3) -> str:
    try:
        return f"{float(v):.{decimals}g}"
    except (ValueError, TypeError):
        return str(v)


def _parse_stats(label: str, info_df: pd.DataFrame) -> dict:
    """Extract per-subcluster stats from subcluster_info."""
    row = info_df[info_df["label"] == label]
    if row.empty:
        return {}
    row = row.iloc[0]

    n_total   = int(row["n_cells"])         if pd.notna(row.get("n_cells"))         else "—"
    n_disease = int(row["n_disease_cells"]) if pd.notna(row.get("n_disease_cells")) else "—"
    n_healthy = (
        n_total - n_disease
        if isinstance(n_total, int) and isinstance(n_disease, int)
        else "—"
    )

    if "disease_composition" in info_df.columns and pd.notna(row.get("disease_composition")):
        disease_composition = str(row["disease_composition"])
    else:
        diseases_str = (
            row.get("diseases_contributing", "")
            if "diseases_contributing" in info_df.columns
            else ""
        )
        if diseases_str and str(diseases_str).strip():
            disease_composition = str(diseases_str)
        else:
            disease_composition = str(row.get("disease", "—"))

    return {
        "n_total":             n_total,
        "n_disease":           n_disease,
        "n_healthy":           n_healthy,
        "fold_enrichment":     _fmt_float(row.get("fold_enrichment", "—")),
        "fdr_pval":            _fmt_float(row.get("pvalue_adj", "—"), decimals=2),
        "cell_type":           str(row.get("subset", "—")),
        "disease_composition": disease_composition,
    }


def _parse_disease_breakdown(
    label: str,
    info_df: pd.DataFrame,
    adata: AnnData,
) -> list[dict]:
    """Build a per-disease breakdown for a subcluster."""
    row = info_df[info_df["label"] == label]
    if row.empty:
        return []
    row = row.iloc[0]

    subcluster_mask = adata.obs[SUBCLUSTER_KEY] == label
    n_subcluster = subcluster_mask.sum()
    if n_subcluster == 0:
        return []

    diseases_str = row.get("diseases_contributing", "") if "diseases_contributing" in info_df.columns else ""
    if diseases_str and str(diseases_str).strip():
        diseases = [d.strip() for d in str(diseases_str).split(",") if d.strip()]
        breakdown = []
        for d in sorted(diseases):
            n = int((subcluster_mask & (adata.obs[DISEASE_KEY] == d)).sum())
            pct = f"{100 * n / n_subcluster:.1f}%"
            breakdown.append({"disease": d, "n_cells": n, "pct": pct, "fold": "—", "pval": "—", "fdr": "—"})
        return breakdown

    disease_label = str(row.get("disease", ""))
    if disease_label and disease_label != "combined":
        n = int(row.get("n_disease_cells", 0))
        pct = f"{100 * n / n_subcluster:.1f}%" if n_subcluster > 0 else "—"
        return [{"disease": disease_label, "n_cells": n, "pct": pct,
                 "fold": _fmt_float(row.get("fold_enrichment", "—")),
                 "pval": _fmt_float(row.get("pvalue", "—")),
                 "fdr":  _fmt_float(row.get("pvalue_adj", "—"), decimals=2)}]
    return []


def _load_de(label: str) -> pd.DataFrame | None:
    """Load DE results CSV(s) for a subcluster and combine if multiple."""
    safe = _safe_label(label)
    de_dir = RESULTS_DIR / "de"
    if not de_dir.exists():
        print(f"  [warn] DE directory not found: {de_dir}")
        return None

    matches = [p for p in sorted(de_dir.glob(f"de_{safe}*.csv"))
               if "all_comparisons" not in p.name]
    if not matches:
        matches = [p for p in sorted(de_dir.glob("de_*.csv"))
                   if safe in p.stem and "all_comparisons" not in p.name]
    if not matches:
        # Backward-compat: NB01 used label.replace("|","_").replace(" ","_")
        _old_safe = _re.sub(r'[^\w\-]', '_', label.replace('|', '_'))
        _old_safe = _re.sub(r'_+', '_', _old_safe).strip('_')
        matches = [p for p in sorted(de_dir.glob(f'de_{_old_safe}*.csv'))
                   if 'all_comparisons' not in p.name]
    if not matches:
        print(f"  [warn] No DE file found for '{label}' in {de_dir}")
        return None

    frames = []
    for p in matches:
        df = pd.read_csv(p)
        df["_comparison"] = p.stem.removeprefix("de_")
        frames.append(df)

    if len(frames) == 1:
        return frames[0].drop(columns=["_comparison"])
    combined = pd.concat(frames, ignore_index=True)
    cols = ["_comparison"] + [c for c in combined.columns if c != "_comparison"]
    return combined[cols]


def _get_de_reference(label: str) -> str:
    """Extract and format the DE reference group label for a subcluster."""
    safe = _safe_label(label)
    de_dir = RESULTS_DIR / "de"
    if not de_dir.exists():
        return "—"
    matches = [p for p in sorted(de_dir.glob(f"de_{safe}*.csv"))
               if "all_comparisons" not in p.name]
    if not matches:
        matches = [p for p in sorted(de_dir.glob("de_*.csv"))
                   if safe in p.stem and "all_comparisons" not in p.name]
    if not matches:
        # Backward-compat: NB01 used label.replace("|","_").replace(" ","_")
        _old_safe = _re.sub(r'[^\w\-]', '_', label.replace('|', '_'))
        _old_safe = _re.sub(r'_+', '_', _old_safe).strip('_')
        matches = [p for p in sorted(de_dir.glob(f'de_{_old_safe}*.csv'))
                   if 'all_comparisons' not in p.name]
    if not matches:
        return "—"

    known_diseases: list[str] = sorted(
        adata.obs[DISEASE_KEY].unique().tolist(), key=len, reverse=True
    )

    def _parse_ref_part(ref_str: str) -> str:
        if not ref_str or ref_str.lower() == "rest":
            return "Rest of dataset"
        for disease in known_diseases:
            if ref_str.endswith(f"_{disease}"):
                cell_type = ref_str[: -len(f"_{disease}")]
                return f"Cell type: {cell_type} | Disease: {disease}"
            if ref_str == disease:
                return f"Disease: {disease}"
        return ref_str

    refs: list[str] = []
    for p in matches:
        ref_str = None
        try:
            df_peek = pd.read_csv(p, nrows=1)
            if "comparison" in df_peek.columns:
                comp = str(df_peek["comparison"].iloc[0])
                if "_vs_" in comp:
                    ref_str = comp.split("_vs_", 1)[1]
        except Exception:
            pass
        if ref_str is None:
            stem = p.stem.removeprefix("de_")
            ref_str = stem.split("_vs_", 1)[1] if "_vs_" in stem else None
        refs.append(_parse_ref_part(ref_str or "rest"))

    seen: set[str] = set()
    unique_refs = [r for r in refs if not (r in seen or seen.add(r))]
    return ", ".join(unique_refs) if unique_refs else "—"


_DE_DISPLAY_COLS = [
    "_comparison", GENE_SYMBOL_COL, "gene_id",
    DE_LFC_COL, "logfoldchanges",
    DE_PVAL_COL, "pvals_adj",
    "pvalue", "pvals",
    "baseMean", "stat",
]

_PVAL_DISPLAY_COLS = {"padj", "pvalue", "pvals_adj", "pvals", DE_PVAL_COL}


def _de_display_df(de_df: pd.DataFrame) -> pd.DataFrame:
    """Return a display-ready DE DataFrame with p-values in scientific notation."""
    show_cols = [c for c in _DE_DISPLAY_COLS if c in de_df.columns]
    if not show_cols:
        show_cols = list(de_df.columns[:8])
    out = de_df[show_cols].copy()
    for c in out.select_dtypes(include="float").columns:
        if c in _PVAL_DISPLAY_COLS:
            out[c] = out[c].apply(lambda v: f"{v:.2e}" if pd.notna(v) else "")
        else:
            out[c] = out[c].round(4)
    return out


print("Helper functions defined.")


## 5. Generate Per-Subcluster Assets (UMAP + Volcano)

In [ ]:
# Pre-compute collision-free safe labels for all subclusters
_build_safe_label_map(subcluster_labels)

subcluster_data = []  # list of dicts, one per subcluster

for label in sorted(subcluster_labels):
    safe = _safe_label(label)
    print(f"\n── {label}")

    card = {
        "label":             label,
        "anchor":            safe,
        "disease":           label.split("|")[1] if "|" in label else label,
        "stats":             _parse_stats(label, subcluster_info),
        "disease_breakdown": _parse_disease_breakdown(label, subcluster_info, adata),
    }

    # Rebuild disease_composition from per-disease breakdown when stored value
    # is a generic placeholder (e.g. "combined").
    _bd = card["disease_breakdown"]
    if _bd:
        _comp = card["stats"].get("disease_composition", "")
        if not _comp or _comp.lower() in ("combined", "—", "unknown", "nan"):
            card["stats"]["disease_composition"] = ", ".join(
                f"{r['disease']}: {r['n_cells']} ({r['pct']})" for r in _bd
            )

    # ── Per-subcluster image directory ────────────────────────────────────
    subcluster_img_dir = REPORT_IMG_DIR / safe
    subcluster_img_dir.mkdir(exist_ok=True)

    # ── UMAP panels ──────────────────────────────────────────────────────
    print("  generating UMAP panels...", end=" ")
    try:
        fig_ct  = _make_umap_celltype(adata, UMAP_FIGSIZE)
        fig_dis = _make_umap_disease(adata, UMAP_FIGSIZE)
        fig_hl  = _make_umap_highlight(adata, label, UMAP_FIGSIZE)

        for key, fig, filename in [
            ("umap_celltype",  fig_ct,  "umap_celltype.png"),
            ("umap_disease",   fig_dis, "umap_disease.png"),
            ("umap_highlight", fig_hl,  "umap_highlight.png"),
        ]:
            fig.savefig(subcluster_img_dir / filename, bbox_inches="tight", dpi=150)
            card[f"{key}_path"]  = f"img/{safe}/{filename}"
            card[f"{key}_bytes"] = _fig_to_bytes(fig)
            plt.close(fig)

        print("OK")
    except Exception as e:
        print(f"FAILED ({e})")
        for key in ["umap_celltype", "umap_disease", "umap_highlight"]:
            card[f"{key}_path"]  = None
            card[f"{key}_bytes"] = None

    # ── Load DE results ───────────────────────────────────────────────────
    de_df = _load_de(label)

    # ── Volcano plot ──────────────────────────────────────────────────────
    print("  generating volcano plot...", end=" ")
    try:
        if de_df is not None and GENE_SYMBOL_COL in de_df.columns:
            df_volcano = format_deseq2_results(
                de_df.set_index(GENE_SYMBOL_COL),
                pval_col=DE_PVAL_COL,
                lfc_col=DE_LFC_COL,
            )
        else:
            df_volcano = de_df
        if df_volcano is not None:
            fig_vol = volcano_plot(
                df=df_volcano,
                pval_cutoff=VOLCANO_PVAL_CUTOFF,
                lfc_cutoff=VOLCANO_LFC_CUTOFF,
                top_n_up=VOLCANO_TOP_N_UP,
                top_n_down=VOLCANO_TOP_N_DOWN,
                figsize=VOLCANO_FIGSIZE,
            )
            fig_vol.savefig(subcluster_img_dir / "volcano.png", bbox_inches="tight", dpi=150)
            card["volcano_path"]  = f"img/{safe}/volcano.png"
            card["volcano_bytes"] = _fig_to_bytes(fig_vol)
            plt.close(fig_vol)
            print("OK")
        else:
            card["volcano_path"]  = None
            card["volcano_bytes"] = None
            print("skipped (no DE data)")
    except Exception as e:
        print(f"FAILED ({e})")
        card["volcano_path"]  = None
        card["volcano_bytes"] = None

    # ── DE DataFrame (stored for HTML render and PPTX) ────────────────────
    card["de_df"] = de_df
    card["stats"]["de_reference"] = _get_de_reference(label)

    subcluster_data.append(card)

print(f"\n✓ Assets generated for {len(subcluster_data)} subclusters.")
print(f"  Images saved to: {REPORT_IMG_DIR}")

## 6. Functional Enrichment (g:Profiler)

In [ ]:
enrich_dir = RESULTS_DIR / "enrichment"
enrich_dir.mkdir(exist_ok=True)

for card in subcluster_data:
    label = card["label"]
    safe  = card["anchor"]
    subcluster_img_dir = REPORT_IMG_DIR / safe
    subcluster_img_dir.mkdir(exist_ok=True)

    enrich_csv_path = enrich_dir / f"enrichment_{safe}.csv"

    print(f"\n── {label}")

    # ── Load or rerun g:Profiler ──────────────────────────────────────────
    enrich_df = None
    if enrich_csv_path.exists() and not FORCE_RERUN_ENRICHMENT:
        print("  loading cached enrichment...", end=" ")
        try:
            enrich_df = pd.read_csv(enrich_csv_path)
            print(f"OK ({len(enrich_df)} terms)")
        except Exception as e:
            print(f"FAILED ({e}); will rerun")
            enrich_df = None

    if enrich_df is None:
        # Build gene list: significant upregulated genes
        de_df = card.get("de_df")
        gene_list = []
        if de_df is not None and not de_df.empty:
            up = de_df.copy()
            if DE_PVAL_COL in up.columns:
                up = up[up[DE_PVAL_COL] < VOLCANO_PVAL_CUTOFF]
            if DE_LFC_COL in up.columns:
                up = up[up[DE_LFC_COL] > VOLCANO_LFC_CUTOFF]
            if GENE_SYMBOL_COL in up.columns:
                gene_list = up[GENE_SYMBOL_COL].dropna().tolist()

        if not gene_list:
            print("  skipped (no significant upregulated genes)")
            card["dotplot_path"]  = None
            card["dotplot_bytes"] = None
            continue

        print(f"  running g:Profiler ({len(gene_list)} genes)...", end=" ")
        try:
            enrich_df = get_enriched_terms(
                gene_list,
                organism=ENRICHMENT_ORGANISM,
                pval_adjust_sign=ENRICHMENT_PVAL,
            )
            enrich_df.to_csv(enrich_csv_path, index=False)
            print(f"OK ({len(enrich_df)} terms)")
        except Exception as e:
            print(f"FAILED ({e})")
            card["dotplot_path"]  = None
            card["dotplot_bytes"] = None
            continue

    if enrich_df is None or enrich_df.empty:
        print("  no enrichment terms found")
        card["dotplot_path"]  = None
        card["dotplot_bytes"] = None
        continue

    # ── Filter and plot ───────────────────────────────────────────────────
    print("  generating enrichment dotplot...", end=" ")
    try:
        _plot_enrich = (
            enrich_df[enrich_df["source"].isin(ENRICHMENT_SOURCES)].copy()
            if ENRICHMENT_SOURCES else enrich_df.copy()
        )
        if "p_value" in _plot_enrich.columns:
            _plot_enrich = _plot_enrich[_plot_enrich["p_value"] < ENRICHMENT_PVAL]
        if _plot_enrich.empty:
            raise ValueError("No enrichment terms pass the filters.")

        fig_dot = create_pathway_dotplot(
            data=_plot_enrich,
            source_colors=PATHWAY_COLORS,
            figsize=ENRICHMENT_FIGSIZE,
            max_pathways=ENRICHMENT_MAX_PATHWAYS,
            title=f"Functional enrichment — {label}",
        )
        fig_dot.savefig(subcluster_img_dir / "enrichment.png", bbox_inches="tight", dpi=150)
        card["dotplot_path"]  = f"img/{safe}/enrichment.png"
        card["dotplot_bytes"] = _fig_to_bytes(fig_dot)
        plt.close(fig_dot)
        print("OK")
    except Exception as e:
        print(f"FAILED ({e})")
        card["dotplot_path"]  = None
        card["dotplot_bytes"] = None

print(f"\n✓ Enrichment complete.  CSVs saved to: {enrich_dir}")

## 6. Render HTML Report

In [ ]:
if "__file__" in dir():
    TEMPLATE_DIR = Path(__file__).parent / "templates"
elif (Path.cwd() / "templates").is_dir():
    TEMPLATE_DIR = Path("templates")
else:
    TEMPLATE_DIR = Path("notebooks/disease_subclusters/templates")

# ── Save DE CSVs and prepare per-card JSON for the lazy DE table ──────────────
de_csv_dir = REPORT_DIR / "de"
de_csv_dir.mkdir(exist_ok=True)

for card in subcluster_data:
    de_df = card.get("de_df")
    if de_df is not None and not de_df.empty:
        if DE_PVAL_COL in de_df.columns:
            sig_df = de_df[de_df[DE_PVAL_COL] < VOLCANO_PVAL_CUTOFF].copy()
        else:
            sig_df = de_df.copy()
        display_df = sig_df if not sig_df.empty else de_df.copy()

        sort_keys, sort_asc = [], []
        if DE_LFC_COL in display_df.columns:
            sort_keys.append(DE_LFC_COL);  sort_asc.append(False)
        if DE_PVAL_COL in display_df.columns:
            sort_keys.append(DE_PVAL_COL); sort_asc.append(True)
        if sort_keys:
            display_df = display_df.sort_values(sort_keys, ascending=sort_asc)

        csv_path = de_csv_dir / f"{card['anchor']}.csv"
        _de_display_df(display_df).to_csv(csv_path, index=False)
        card["de_csv_path"] = f"de/{card['anchor']}.csv"
        card["de_json"]     = _de_display_df(display_df).to_json(orient="split")
        n_sig = len(sig_df)
    else:
        card["de_json"]     = None
        card["de_csv_path"] = None
        n_sig = 0

    print(f"  {card['label']}: {n_sig} significant genes in HTML table")

# ── Render template ────────────────────────────────────────────────────────────
env      = Environment(loader=FileSystemLoader(str(TEMPLATE_DIR)), autoescape=False)
template = env.get_template("subcluster_report.html.j2")

html_out = template.render(
    tissue=TISSUE,
    subclusters=subcluster_data,
    generated_at=datetime.now().strftime("%Y-%m-%d %H:%M"),
)

html_path = REPORT_DIR / "index.html"
html_path.write_text(html_out, encoding="utf-8")
size_kb = html_path.stat().st_size / 1024
print(f"\n✓ HTML report written: {html_path}  ({size_kb:.0f} KB)")
print(f"  DE CSVs written to : {de_csv_dir}")
print(f"  Open in browser    : {html_path.resolve()}")

## 7. Render PowerPoint Deck

In [ ]:

# ── Layout constants ─────────────────────────────────────────────────────────
SLIDE_W = Inches(13.33)
SLIDE_H = Inches(7.5)

_DARK  = RGBColor(0x1a, 0x1a, 0x2e)
_BLUE  = RGBColor(0x4a, 0x90, 0xd9)
_WHITE = RGBColor(0xff, 0xff, 0xff)
_LGREY = RGBColor(0xf0, 0xf2, 0xf5)
_TEXT  = RGBColor(0x1a, 0x1a, 0x2e)

# Left panel (stats + DE table)
_LEFT_L = Cm(0.4)
_LEFT_W = Cm(12.7)
# Right panel (images)
_RIGHT_L = Cm(13.5)


def _add_text_box(slide, left, top, width, height, text, *, bold=False,
                  fontsize=10, color=_TEXT, align=PP_ALIGN.LEFT, wrap=True):
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf    = txBox.text_frame
    tf.word_wrap = wrap
    p      = tf.paragraphs[0]
    p.text = text
    p.alignment = align
    run = p.runs[0] if p.runs else p.add_run()
    run.font.bold = bold
    run.font.size = Pt(fontsize)
    run.font.color.rgb = color
    return txBox


def _add_image_bytes(slide, img_bytes, left, top, width, height=None):
    """Add a PNG from bytes to the slide. Omit ``height`` to preserve aspect ratio."""
    if img_bytes is None:
        return
    buf = io.BytesIO(img_bytes)
    slide.shapes.add_picture(buf, left, top, width=width, height=height)


def _add_title_bar(slide, label, disease):
    bar = slide.shapes.add_shape(
        1,  # MSO_SHAPE_TYPE.RECTANGLE
        0, 0, SLIDE_W, Cm(1.8),
    )
    bar.fill.solid()
    bar.fill.fore_color.rgb = _DARK
    bar.line.fill.background()

    _add_text_box(
        slide, Cm(0.4), Cm(0.1), Cm(20), Cm(1.6),
        label, bold=True, fontsize=14, color=_WHITE,
    )
    _add_text_box(
        slide, Cm(21), Cm(0.1), Cm(12), Cm(1.6),
        f"Disease: {disease}", bold=False, fontsize=11, color=_BLUE,
        align=PP_ALIGN.RIGHT,
    )


def _set_cell_text(cell, text: str, *, bold: bool = False, fontsize: int = 6,
                   bg_color: RGBColor | None = None,
                   color: RGBColor = _TEXT):
    """Set text and font on a pptx table cell (explicit color prevents theme inheritance)."""
    if bg_color is not None:
        cell.fill.solid()
        cell.fill.fore_color.rgb = bg_color
    tf = cell.text_frame
    tf.word_wrap = False
    p = tf.paragraphs[0]
    p.clear()
    run = p.add_run()
    run.text = text
    run.font.bold = bold
    run.font.size = Pt(fontsize)
    run.font.color.rgb = color


def _add_table_section_title(slide, left, top, width, text):
    """Add a small uppercase section label above a table; returns new top."""
    _add_text_box(slide, left, top, width, Cm(0.42),
                  text, bold=True, fontsize=7, color=_BLUE)
    return top + Cm(0.42)


def _add_stats_table(slide, stats, breakdown, top=Cm(2.0),
                     left=Cm(0.4), width=Cm(12.7)):
    """Render subcluster stats as a two-column pptx table with a labelled title.

    Always shows the disease_composition string (rebuilt from breakdown in
    the asset loop). Per-disease breakdown rows are appended below it.
    Returns the bottom y-coordinate so the next element can be placed below.
    """
    rows_data = [
        ("Total cells",        str(stats.get("n_total",         "—"))),
        ("Disease cells",      str(stats.get("n_disease",       "—"))),
        ("Healthy cells",      str(stats.get("n_healthy",       "—"))),
        ("Fold enrichment",    str(stats.get("fold_enrichment", "—"))),
        ("FDR p-value",        str(stats.get("fdr_pval",        "—"))),
        ("Cell type",          str(stats.get("cell_type",       "—"))),
        ("Disease composition",str(stats.get("disease_composition", "—"))),
        ("DE reference",        str(stats.get("de_reference",        "—"))),
    ]
    # Per-disease breakdown rows (indented)
    if breakdown:
        for r in breakdown:
            rows_data.append((
                f"  \u21b3 {r['disease']}",
                f"n={r['n_cells']} ({r['pct']})"
                + (f"  FDR={r['fdr']}" if r['fdr'] != "—" else ""),
            ))

    tbl_top = _add_table_section_title(slide, left, top, width, "SUMMARY STATISTICS")
    row_h   = Cm(0.42)
    n_rows  = len(rows_data)
    tbl     = slide.shapes.add_table(n_rows, 2, left, tbl_top, width, row_h * n_rows).table

    tbl.columns[0].width = Cm(4.5)
    tbl.columns[1].width = width - Cm(4.5)

    for ri, (lbl, val) in enumerate(rows_data):
        _set_cell_text(tbl.cell(ri, 0), lbl, bold=True,  fontsize=6, bg_color=_LGREY)
        _set_cell_text(tbl.cell(ri, 1), val, bold=False, fontsize=6)

    return tbl_top + row_h * n_rows   # bottom y


def _add_de_table(slide, de_df, top, left=Cm(0.4), width=Cm(12.7)):
    """Render top-N significant upregulated DE genes as a pptx table with title."""
    if de_df is None or de_df.empty:
        return

    # Filter to significant upregulated genes only
    up_df = de_df.copy()
    if DE_PVAL_COL in up_df.columns:
        up_df = up_df[up_df[DE_PVAL_COL] < VOLCANO_PVAL_CUTOFF]
    if DE_LFC_COL in up_df.columns:
        up_df = up_df[up_df[DE_LFC_COL] > VOLCANO_LFC_CUTOFF]
    if DE_PVAL_COL in up_df.columns:
        up_df = up_df.sort_values(DE_PVAL_COL)
    if up_df.empty:
        return

    tbl_top = _add_table_section_title(slide, left, top, width, "TOP UPREGULATED GENES")

    preferred = [GENE_SYMBOL_COL, DE_LFC_COL, DE_PVAL_COL, "baseMean", "gene_id"]
    show_cols = [c for c in preferred if c in up_df.columns][:5]
    if not show_cols:
        show_cols = list(up_df.columns[:5])

    sub = up_df[show_cols].head(PPTX_TOP_DE_GENES).fillna("").copy()
    _pval_cols = {DE_PVAL_COL, "padj", "pvalue", "pvals_adj", "pvals"}
    for c in sub.select_dtypes(include="float").columns:
        if c in _pval_cols:
            sub[c] = sub[c].apply(
                lambda v: f"{v:.2e}" if pd.notna(v) and v != "" else ""
            )
        else:
            sub[c] = sub[c].round(4).astype(str)

    n_rows = len(sub) + 1   # +1 header row
    n_cols = len(show_cols)
    row_h  = Cm(0.42)
    tbl    = slide.shapes.add_table(n_rows, n_cols, left, tbl_top, width, row_h * n_rows).table

    col_w = width // n_cols
    for ci in range(n_cols):
        tbl.columns[ci].width = col_w

    for ci, col in enumerate(show_cols):
        _set_cell_text(tbl.cell(0, ci), col, bold=True, fontsize=6, bg_color=_LGREY)

    for ri, (_, row) in enumerate(sub.iterrows(), start=1):
        for ci, col in enumerate(show_cols):
            _set_cell_text(tbl.cell(ri, ci), str(row[col]), fontsize=6)


# ── Build presentation ───────────────────────────────────────────────────────
prs = Presentation()
prs.slide_width  = SLIDE_W
prs.slide_height = SLIDE_H
blank_layout = prs.slide_layouts[6]   # blank

for card in subcluster_data:
    slide = prs.slides.add_slide(blank_layout)

    _add_title_bar(slide, card["label"], card["disease"])

    # ── Left panel: stats table then DE table ─────────────────────────────
    stats_bottom = _add_stats_table(slide, card["stats"], card["disease_breakdown"])
    _add_de_table(slide, card.get("de_df"), top=stats_bottom + Cm(1.2))

    # ── Right panel: images ───────────────────────────────────────────────
    # Row 1: 3× UMAP — taller and wider to fill available space
    umap_top = Cm(2.0)
    umap_h   = Cm(7.2)
    umap_w   = Cm(6.5)
    gap      = Cm(0.18)

    for i, key in enumerate(["umap_celltype_bytes", "umap_disease_bytes", "umap_highlight_bytes"]):
        _add_image_bytes(slide, card.get(key),
                         _RIGHT_L + i * (umap_w + gap), umap_top, umap_w)

    # Row 2: volcano + dotplot — preserve aspect ratio
    plot_top = umap_top + umap_h + Cm(0.25)
    plot_w   = Cm(10.0)

    _add_image_bytes(slide, card.get("volcano_bytes"),
                     _RIGHT_L,               plot_top, plot_w)
    _add_image_bytes(slide, card.get("dotplot_bytes"),
                     _RIGHT_L + plot_w + gap, plot_top, plot_w)

pptx_path = OUTPUT_DIR / f"{TISSUE}_subcluster_cards.pptx"
prs.save(str(pptx_path))
print(f"✓ PowerPoint deck written: {pptx_path}  ({pptx_path.stat().st_size / 1024:.0f} KB)")


## Summary

| Output | Path |
|--------|------|
| HTML report | `{REPORT_DIR}/index.html` |
| Per-subcluster images | `{REPORT_DIR}/img/{safe}/` |
| DE tables (CSV) | `{REPORT_DIR}/de/{safe}.csv` |
| Enrichment CSVs | `{RESULTS_DIR}/enrichment/enrichment_{safe}.csv` |
| PowerPoint deck | `{OUTPUT_DIR}/{TISSUE}_subcluster_cards.pptx` |

**Viewing the report:** open `index.html` in a browser — the `img/` and `de/` folders must remain alongside it.  
**Sharing:** zip the entire `{REPORT_DIR}/` directory; all images and CSVs are referenced by relative paths.  
**Next step:** run `03_consensus_gene_list.ipynb` to derive a cross-dataset consensus gene signature.